# Iraq Farms Health: copy the published crop map from Drive to Cloudflare R2

Copies `publish/2026/` from Drive to the R2 bucket `ifh-data`, served at **https://data.iraqfarmshealth.com/crops/2026/**. It uses `rclone sync`: only new or changed files are uploaded, and files no longer in the publish folder are deleted from that one R2 folder (nothing else in the bucket is touched). Run it **after** the publish notebook and the 2021 comparison.

**Keys (once).** In Cloudflare, go to R2 Object Storage → *Manage API tokens* → *Create Account API token*: permission **Object Read & Write**, applied to **ifh-data only**. Copy the three values it shows into Colab's **🔑 Secrets** panel (left bar) with these names, and switch *Notebook access* on for this notebook:
- `R2_ACCESS_KEY_ID`
- `R2_SECRET_ACCESS_KEY`
- `R2_ENDPOINT`: the S3 endpoint, `https://<account id>.r2.cloudflarestorage.com` (without `/ifh-data`)

The keys never appear in the notebook or its output. Never paste them anywhere else.

**Runtime ▸ Run all.** Step 3 shows what would change and stops there unless `GO = True`. Step 5 checks every file in R2 against Drive by size and MD5.

In [ ]:
# 1. Settings and Drive
RUN_NAME    = "iraq_winter2026"
DATA_FOLDER = "2026"          # publish/<DATA_FOLDER>/ in Drive  ->  crops/<DATA_FOLDER>/ in R2
GO          = False           # step 3 only previews the changes; set True and run steps 3-5 again to upload

from google.colab import drive
drive.mount("/content/drive")
import os, json, subprocess
RUN  = f"/content/drive/MyDrive/iraq-farms-health/worldcereal/{RUN_NAME}"
SRC  = f"{RUN}/publish/{DATA_FOLDER}"
DEST = f"r2:ifh-data/crops/{DATA_FOLDER}"
idx = json.load(open(f"{SRC}/index.json"))
files = [os.path.join(d, f) for d, _, fs in os.walk(SRC) for f in fs]
print(f"Source: {SRC}\n  {len(files)} files, {sum(os.path.getsize(f) for f in files) / 1e6:.1f} MB")
print(f"  index.json: generated {idx.get('generated_utc')}, {len(idx['processed'])} tiles, {len(idx['cells'])} cells")
print(f"  compare_2021.json present: {os.path.exists(f'{SRC}/compare_2021.json')}")
print(f"Destination: {DEST}  (https://data.iraqfarmshealth.com/crops/{DATA_FOLDER}/)")

In [ ]:
# 2. rclone, configured from the Colab secrets (nothing is written to disk or printed)
!curl -s https://rclone.org/install.sh | sudo bash > /dev/null 2>&1; rclone version | head -1
from google.colab import userdata
os.environ.update({
    "RCLONE_CONFIG_R2_TYPE": "s3",
    "RCLONE_CONFIG_R2_PROVIDER": "Cloudflare",
    "RCLONE_CONFIG_R2_ACCESS_KEY_ID": userdata.get("R2_ACCESS_KEY_ID"),
    "RCLONE_CONFIG_R2_SECRET_ACCESS_KEY": userdata.get("R2_SECRET_ACCESS_KEY"),
    "RCLONE_CONFIG_R2_ENDPOINT": userdata.get("R2_ENDPOINT").rstrip("/"),
    "RCLONE_CONFIG_R2_NO_CHECK_BUCKET": "true",
})
r = subprocess.run(["rclone", "lsf", "r2:ifh-data", "--max-depth", "1"], capture_output=True, text=True)
print("Bucket reachable:", r.returncode == 0, "| top level:", r.stdout.split() or "(empty)")
if r.returncode: print(r.stderr[-800:])

In [ ]:
# 3. Preview (GO = False) or upload (GO = True). Files under 200 MB go up in one part, so R2 keeps their MD5 for step 5.
FLAGS = ["--checksum", "--s3-upload-cutoff=200M", "--transfers=8", "--stats-one-line", "--stats=10s"]
if not GO:
    r = subprocess.run(["rclone", "sync", SRC, DEST, "--dry-run", *FLAGS], capture_output=True, text=True)
    lines = [l for l in r.stderr.splitlines() if "Skipped" in l]
    print(f"Would change {len(lines)} file(s):")
    for l in lines[:40]: print("  ", l.split("NOTICE: ")[-1])
    if len(lines) > 40: print(f"   ... and {len(lines) - 40} more")
    print("\nNothing uploaded. To upload: set GO = True in step 1, then run steps 1-5 again.")
else:
    !rclone sync "$SRC" "$DEST" {" ".join(FLAGS)}

In [ ]:
# 4. What is in R2 now
!rclone size "$DEST"

In [ ]:
# 5. Check every file: R2 against Drive, by size and MD5 (reads the whole folder once)
r = subprocess.run(["rclone", "check", SRC, DEST], capture_output=True, text=True)
print(r.stderr[-1500:])
print("ALL FILES MATCH" if r.returncode == 0 else "DIFFERENCES FOUND: see above")

In [ ]:
# 6. Optional, once: remove the test folder created while setting up the bucket
REMOVE_TEST_FOLDER = False
if REMOVE_TEST_FOLDER:
    !rclone purge r2:ifh-data/test && echo "test/ removed"